# Validation: AFT regression with time-varying covariates (`tvaft.py`)

The TFA model asks how behaviour speeds up or slows down the days from an account's first action to its first accusation,
with behaviour that changes as the account acts. `lifelines` has no AFT fitter with time-varying covariates, so
`tvaft.TimeVaryingAFT` implements one (Cox & Oakes 1984, ch. 6): each account's clock runs at `exp(-x(t)·β)` times
baseline speed, and `exp(β)` is a time ratio.

Checks:
1. constant covariates against `lifelines` (Weibull, log-normal, log-logistic; censoring and delayed entry);
2. constant covariates against R `eha::aftreg`;
3. time-varying covariates against installed `eha` 2.12.0: they disagree, because eha's event-density term uses the
   current-covariate scaled time instead of the accumulated clock. Reproducing eha's formula and integrating its density
   shows it is not a proper density;
4. time-varying covariates against `eha` with that one term patched (`eha_aftreg_tv_fix.patch`, built in a temp dir);
5. Monte Carlo on simulated accounts with known time ratios (`tvaft_simulation.py`): bias and 95% CI coverage, model-based and
   robust standard errors, against an AFT that uses each covariate's value at the event;
6. generalized gamma on Weibull data, and delayed entry with censoring.

Needs R with `eha` in `R_LIBS_USER`. Results go to `tv_aft_validation_output/` through `save_new()`.

In [ ]:
# Libraries, including the project's time-varying AFT fitter (tvaft.py) and its simulator (tvaft_simulation.py).
import pandas as pd
import numpy as np
import os
import re
import subprocess
import tempfile
import time
import warnings
import requests
from scipy import integrate, optimize
import autograd.numpy as anp
from autograd import grad
from lifelines import WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter
from tvaft import TimeVaryingAFT, baseline_logf_logS, split_at
from tvaft_simulation import simulate, TRUE_TIME_RATIOS
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Output folder, Monte Carlo replications, covariate names, and the three baseline distributions checked.
OUT_DIR = 'tv_aft_validation_output'
N_REPS = 200
COVS = ['x1', 'x2', 'x3']
DISTS = ['weibull', 'lognormal', 'loglogistic']
TMP = tempfile.mkdtemp(prefix='tvaft_validation_')      # CSV exchange with R and the patched eha build
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# Fit the same data in R with eha::aftreg (tvaft_eha_check.R) via CSV files; eha_lib points R at the patched eha build.
def run_eha(episodes, subjects, dist, eha_lib=None):
    ep_csv, sb_csv, out_csv = (os.path.join(TMP, f) for f in ('ep.csv', 'sb.csv', 'eha_out.csv'))
    episodes.to_csv(ep_csv, index=False)
    subjects.reset_index().to_csv(sb_csv, index=False)
    env = dict(os.environ, **({'EHA_LIB': eha_lib} if eha_lib else {}))
    r = subprocess.run(['Rscript', 'tvaft_eha_check.R', ep_csv, sb_csv, dist, out_csv], capture_output=True, text=True, env=env)
    if r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return pd.read_csv(out_csv).set_index('term')

# Side-by-side coefficients, SEs and log-likelihoods from Python and R for one check.
def compare(py, R, label):
    return pd.DataFrame({'check': label, 'term': COVS, 'coef_python': py.summary.loc[COVS, 'coef'].to_numpy(),
                         'coef_R': R.loc[COVS, 'coef'].to_numpy(), 'se_python': py.summary.loc[COVS, 'se'].to_numpy(),
                         'se_R': R.loc[COVS, 'se'].to_numpy(), 'loglik_python': py.loglik_, 'loglik_R': R['loglik'].iloc[0]})

## 1. Constant covariates vs `lifelines` (with censoring and delayed entry)

In [ ]:
# Check 1: with constant covariates, tvaft must match lifelines exactly. Simulated data: 800 subjects,
# log-time linear in the covariates, random right-censoring, and delayed entry at day 0.5 for 30%.
rng = np.random.default_rng(1)
n = 800
x = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.binomial(1, .4, n), 'x3': rng.normal(size=n)})
T = np.exp(np.log(20) + x.to_numpy() @ np.log(TRUE_TIME_RATIOS) + 2.0 * np.log(rng.exponential(size=n)))
C = rng.uniform(5, 2000, n)
entry = np.where(rng.random(n) < .3, 0.5, 0.0)
keep = T > entry                                            # delayed entry: only subjects still at risk at entry
fixed = x[keep].assign(T=np.minimum(T, C)[keep], E=(T <= C)[keep].astype(int), entry=entry[keep]).reset_index(drop=True)
fixed_sub = pd.DataFrame({'event': fixed['E'], 'entry': fixed['entry']}, index=fixed.index)
# The same data as counting-process episodes, split at the entry time.
fixed_ep = fixed[COVS].assign(id=fixed.index, start=0.0, stop=fixed['T'])[['id', 'start', 'stop'] + COVS]
fixed_ep = split_at(fixed_ep, fixed_sub['entry'][fixed_sub['entry'] > 0])
print(f"n={len(fixed)}, events={fixed['E'].sum()}, delayed entry={int((fixed['entry'] > 0).sum())}")

# Fit each distribution with lifelines and with tvaft; lifelines names the scale parameter lambda_, mu_ or alpha_.
rows = []
for dist, LF in zip(DISTS, [WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter]):
    lf = LF().fit(fixed[COVS + ['T', 'E', 'entry']], 'T', 'E', entry_col='entry')
    py = TimeVaryingAFT(dist).fit(fixed_ep, fixed_sub, COVS)
    key = [k for k in ('lambda_', 'mu_', 'alpha_') if k in lf.params_.index.get_level_values(0)][0]
    rows.append(pd.DataFrame({'dist': dist, 'term': COVS, 'coef_lifelines': lf.params_[key][COVS].to_numpy(),
                              'coef_tvaft': py.summary.loc[COVS, 'coef'].to_numpy(),
                              'se_lifelines': lf.standard_errors_[key][COVS].to_numpy(),
                              'se_tvaft': py.summary.loc[COVS, 'se'].to_numpy(),
                              'loglik_lifelines': lf.log_likelihood_, 'loglik_tvaft': py.loglik_}))
check1 = pd.concat(rows, ignore_index=True)
check1

## 2. Constant covariates vs R `eha::aftreg`

In [ ]:
# Check 2: with constant covariates, tvaft must match R eha::aftreg. Print the installed eha version (used again in check 4).
eha_version = subprocess.run(['Rscript', '-e', 'cat(as.character(packageVersion("eha", lib.loc=Sys.getenv("R_LIBS_USER"))))'],
                             capture_output=True, text=True).stdout.strip()
print('installed eha', eha_version)
# Simulated accounts with time-varying covariates; collapse to one episode per account with the final values.
sim_ep, sim_sub = simulate(1000, np.random.default_rng(42))       # time-varying data, used again in sections 3-4
const_ep = (sim_ep.groupby('id').agg(start=('start', 'min'), stop=('stop', 'max'),
                                     **{c: (c, 'last') for c in COVS}).reset_index())   # final values held constant
check2 = pd.concat([compare(TimeVaryingAFT(d).fit(const_ep, sim_sub, COVS), run_eha(const_ep, sim_sub, d), d)
                    for d in DISTS], ignore_index=True)
check2

## 3. Time-varying covariates vs installed `eha`

Same simulated accounts, now with covariates that change. Installed `eha` gives different answers. Its Weibull event term
(`src/aftreg.c`) is `log γ − (α − zβ) + (γ − 1)(log t − (α − zβ)) + log h0(clock^γ)`: the `(γ − 1)` term uses `t` scaled by the
*current* covariates rather than the accumulated clock. Coding that formula in Python reproduces eha's fit, and integrating its
density for one account whose covariate switches on shows the total probability is not 1.

In [ ]:
# Check 3: time-varying covariates against the installed eha (the two are expected to differ).
check3 = pd.concat([compare(TimeVaryingAFT(d).fit(sim_ep, sim_sub, COVS), run_eha(sim_ep, sim_sub, d), d)
                    for d in DISTS], ignore_index=True)
display(check3)

# Reproduce eha's Weibull likelihood in Python to show the gap comes from eha's event term, not from tvaft:
# eha evaluates the hazard at t*exp(-linear predictor) instead of on the accumulated clock.
X = sim_ep[COVS].to_numpy(); dur = (sim_ep['stop'] - sim_ep['start']).to_numpy(); t_abs = sim_ep['stop'].to_numpy()
fi = np.flatnonzero(~sim_ep['id'].duplicated().to_numpy()); li = np.r_[fi[1:] - 1, len(sim_ep) - 1]
ev = sim_sub.loc[sim_ep['id'].to_numpy()[li], 'event'].to_numpy(float)
def eha_weibull_negll(p):                                         # eha aftreg.c Weibull branch, lifeExp signs
    beta, mu, log_shape = p[:3], p[3], p[4]
    g = anp.exp(log_shape); amb = mu + anp.dot(X, beta)
    cs = anp.cumsum(dur * anp.exp(-amb)); clock = cs[li] - anp.concatenate([anp.zeros(1), cs[:-1]])[fi]
    return -anp.sum(ev * (log_shape - amb[li] + (g - 1) * (anp.log(t_abs[li]) - amb[li])) - clock ** g)
r = optimize.minimize(eha_weibull_negll, np.r_[0, 0, 0, 2.5, -0.7], jac=grad(eha_weibull_negll), method='BFGS')
R_w = check3[check3['check'] == 'weibull']
print('eha formula in Python: loglik %.4f (eha %.4f), coefs %s (eha %s)'
      % (-r.fun, R_w['loglik_R'].iloc[0], np.round(r.x[:3], 4), np.round(R_w['coef_R'].to_numpy(), 4)))

# Integrate both event densities for one account whose covariate switches on at day 5. A valid density integrates to 1;
# the eha formula does not.
mu, g, beta, s = np.log(15), 0.45, np.log(0.4), 5.0               # covariate switches on at day 5, time ratio 0.4
clock = lambda t: (t if t < s else s + (t - s) / 0.4) * np.exp(-mu)
amb = lambda t: mu + (beta if t >= s else 0.0)
densities = {'tvaft (accumulated clock)': lambda t: g * clock(t) ** (g - 1) * np.exp(-amb(t)) * np.exp(-clock(t) ** g),
             'eha 2.12.0 formula': lambda t: g * (t * np.exp(-amb(t))) ** (g - 1) * np.exp(-amb(t)) * np.exp(-clock(t) ** g)}
density_check = pd.DataFrame({'model': list(densities),
                              'total_probability': [sum(integrate.quad(f, a, b, limit=500)[0]
                                                        for a, b in [(0, s), (s, 1e3), (1e3, 1e7), (1e7, np.inf)])
                                                    for f in densities.values()]})
density_check

## 4. Time-varying covariates vs `eha` with the one-term patch

In [ ]:
# Check 4: download the installed eha version from CRAN (the Archive if it is no longer current), apply the
# one-term patch (eha_aftreg_tv_fix.patch), install it in a temporary library, and refit; tvaft and patched eha should agree.
# Download with up to 10 retries.
def download(url, path):
    retry_count = 0
    while True:
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            open(path, 'wb').write(r.content)
            return
        except Exception:
            retry_count += 1
            if retry_count >= 10:
                raise
            time.sleep(5)

tarball = os.path.join(TMP, f'eha_{eha_version}.tar.gz')
try:
    download(f'https://cloud.r-project.org/src/contrib/eha_{eha_version}.tar.gz', tarball)
except Exception:
    download(f'https://cloud.r-project.org/src/contrib/Archive/eha/eha_{eha_version}.tar.gz', tarball)
patched_lib = os.path.join(TMP, 'Rlib_eha_patched'); os.makedirs(patched_lib)
subprocess.run(['tar', 'xzf', tarball, '-C', TMP], check=True)
subprocess.run(['patch', '-p1', '-d', os.path.join(TMP, 'eha'), '-i', os.path.abspath('eha_aftreg_tv_fix.patch')], check=True)
build = subprocess.run(['R', 'CMD', 'INSTALL', '--no-test-load', '-l', patched_lib, os.path.join(TMP, 'eha')], capture_output=True, text=True)
print(build.stdout.strip().splitlines()[-1])

check4 = pd.concat([compare(TimeVaryingAFT(d).fit(sim_ep, sim_sub, COVS), run_eha(sim_ep, sim_sub, d, patched_lib), d)
                    for d in DISTS], ignore_index=True)
check4

## 5. Monte Carlo: recovery and coverage (N_REPS simulated samples of 1,000 accounts)

True time ratios: x1 fixed trait 0.70, x2 switches on 0.40, x3 log(1 + actions so far) 1.25. "Value at event" is an ordinary
Weibull AFT using each covariate's value when the account is accused, the bias the time-varying model avoids.

In [ ]:
# Check 5: Monte Carlo. For each replication, simulate 1,000 accounts, fit tvaft (Weibull, robust SEs), and also fit the
# naive model that uses each covariate's value at the event time. Compare bias and 95% CI coverage against the true time ratios.
rows = []
for rep in range(N_REPS):
    ep, sub = simulate(1000, np.random.default_rng(1000 + rep))
    m = TimeVaryingAFT('weibull').fit(ep, sub, COVS, robust=True)
    b = m.params_[:3]
    se_model = np.sqrt(np.diag(m.cov_model_))[:3]; se_robust = np.sqrt(np.diag(m.cov_))[:3]
    last = ep.groupby('id').tail(1).set_index('id')
    naive = WeibullAFTFitter().fit(last[COVS].assign(T=last['stop'], E=sub['event']), 'T', 'E')
    for k, c in enumerate(COVS):
        rows.append({'rep': rep, 'term': c, 'coef': b[k], 'se_model': se_model[k], 'se_robust': se_robust[k],
                     'coef_value_at_event': naive.params_['lambda_'][c]})
mc = pd.DataFrame(rows)
truth = dict(zip(COVS, np.log(TRUE_TIME_RATIOS)))
# Whether each 95% CI (model-based and robust) covers the true coefficient.
mc['covered_model'] = (mc['coef'] - 1.96 * mc['se_model'] <= mc['term'].map(truth)) & (mc['term'].map(truth) <= mc['coef'] + 1.96 * mc['se_model'])
mc['covered_robust'] = (mc['coef'] - 1.96 * mc['se_robust'] <= mc['term'].map(truth)) & (mc['term'].map(truth) <= mc['coef'] + 1.96 * mc['se_robust'])
mc_summary = mc.groupby('term').agg(mean_coef=('coef', 'mean'), sd_coef=('coef', 'std'), mean_se_model=('se_model', 'mean'),
                                    mean_se_robust=('se_robust', 'mean'), coverage_model=('covered_model', 'mean'),
                                    coverage_robust=('covered_robust', 'mean'), mean_coef_value_at_event=('coef_value_at_event', 'mean'))
mc_summary.insert(0, 'true_time_ratio', TRUE_TIME_RATIOS)
mc_summary.insert(1, 'time_ratio_tvaft', np.exp(mc_summary['mean_coef']))
mc_summary['time_ratio_value_at_event'] = np.exp(mc_summary['mean_coef_value_at_event'])
print(f'{N_REPS} reps; Monte Carlo SE of a 95% coverage rate ~{np.sqrt(.95 * .05 / N_REPS):.3f}')
mc_summary

## 6. Generalized gamma on Weibull data; delayed entry with censoring

In [ ]:
# Check 6a: AIC across distributions on Weibull data, and the generalized gamma shape q (q = 1 is Weibull).
fits = {d: TimeVaryingAFT(d).fit(sim_ep, sim_sub, COVS) for d in DISTS + ['gengamma']}
aic = pd.DataFrame({'AIC': {d: f.AIC_ for d, f in fits.items()}})
q = fits['gengamma'].summary.loc['q']
print(f"generalized gamma q = {q['coef']:.3f} (95% CI {q['coef'] - 1.96 * q['se']:.3f} to {q['coef'] + 1.96 * q['se']:.3f}); q = 1 is Weibull, q = 0 log-normal")
display(aic)

# Check 6b: delayed entry at day 1 plus censoring at day 60; estimated time ratios should cover the truth.
ep, sub = simulate(1500, np.random.default_rng(7), admin=60.0)   # censored at day 60
at_risk = ep.groupby('id')['stop'].max().loc[lambda s: s > 1].index
ep = split_at(ep[ep['id'].isin(at_risk)], pd.Series(1.0, index=at_risk))
sub = sub.loc[at_risk].assign(entry=1.0)                          # enter at day 1
m = TimeVaryingAFT('weibull').fit(ep, sub, COVS, robust=True)
entry_check = m.summary.loc[COVS, ['time_ratio', 'lower95', 'upper95']].assign(true_time_ratio=TRUE_TIME_RATIOS)
print(f"n={len(sub)} entering at day 1, accused by day 60: {sub['event'].mean():.0%}")
entry_check

## Save

In [ ]:
# Save every check's table.
save_new(check1, 'check1_constant_vs_lifelines.parquet')
save_new(check2, 'check2_constant_vs_eha.parquet')
save_new(check3, 'check3_timevarying_vs_eha_installed.parquet')
save_new(density_check, 'check3_density_integral.parquet')
save_new(check4, 'check4_timevarying_vs_eha_patched.parquet')
save_new(mc, 'check5_monte_carlo_reps.parquet')
save_new(mc_summary.reset_index(), 'check5_monte_carlo_summary.parquet')
save_new(aic.reset_index(names='dist'), 'check6_aic.parquet')
save_new(entry_check.reset_index(names='term'), 'check6_delayed_entry.parquet')